# Hugging Face Pipeline Applications

1. Execute a atividade em grupo
2. Postar a atividade [aqui](https://drive.google.com/drive/folders/1LRpP7Vqs4nSa1fxwAphysI6AHjURnwIF?usp=sharing)
3. Empregue somente modelos abertos



### Alunos

1. Rodrigo Madera
2. Sara Pires
3. Tiago Carvalho
4. Daianny Queiroz


## 0. Preparação do ambiente

Instalação dos pacotes (os mesmos da aula) e definição do dispositivo de execução.
O notebook roda tanto no Colab (GPU CUDA) quanto localmente (CPU ou Apple Silicon/MPS).
Os modelos generativos usam o acelerador disponível; os modelos menores (encoders BERT) rodam em CPU mesmo.


In [ ]:
%pip install -q transformers accelerate sentencepiece protobuf "jinja2>=3.1.6"

In [ ]:
import gc
import torch
import numpy as np
import pandas as pd
import transformers
from transformers import pipeline

transformers.logging.set_verbosity_error()  # silencia os relatórios de carregamento

print("Versão do transformers:", transformers.__version__)

# Dispositivo: GPU (Colab), MPS (Apple Silicon) ou CPU
if torch.cuda.is_available():
    DEVICE, DTYPE = "cuda", torch.float16
elif torch.backends.mps.is_available():
    DEVICE, DTYPE = "mps", torch.float16
else:
    DEVICE, DTYPE = "cpu", torch.float32

print("Dispositivo:", DEVICE, "| Precisão:", DTYPE)

def liberar_memoria():
    """Libera RAM/VRAM entre um modelo e outro (essencial em máquinas com pouca memória)."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()

# 1. Task: Chat com contexto (`Text-Generation`)

Empregando ao menos um modelo diferente do empregado em sala, implemente e compare o resultado de dois modelos de Chat com contexto. Discuta em poucas linhas o resultado. Chat em português.

Modelos comparados (ambos abertos e leves o suficiente para o Colab gratuito ou execução local):

| | Modelo | Parâmetros | Observação |
|--|--|--|--|
| A | [`Qwen/Qwen2.5-1.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) | 1,5 B | multilíngue; linha de base vista em sala |
| B | [`TinyLlama/TinyLlama-1.1B-Chat-v1.0`](https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0) | 1,1 B | **diferente do visto em sala**; treinado quase só em inglês |

O experimento é idêntico para os dois: uma conversa de **3 turnos em português** sobre um roteiro de viagem, reenviando o histórico completo a cada turno. O 3º turno — *"quantos dias tem o roteiro e para qual cidade?"* — só pode ser respondido por quem **manteve o contexto**. Ao final, repetimos a pergunta do 3º turno **sem** o histórico, como controle.

In [ ]:
MODELO_A = "Qwen/Qwen2.5-1.5B-Instruct"          # linha de base (empregado em sala)
MODELO_B = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # modelo diferente do empregado em sala

SISTEMA = ("Você é um assistente prestativo que responde sempre em português do Brasil, "
           "de forma clara e concisa.")

TURNOS = [
    "Olá! Estou montando um roteiro de 3 dias para conhecer Fortaleza. Pode me ajudar com uma sugestão por dia?",
    "No segundo dia eu prefiro praias mais tranquilas. O que você sugere mudar?",
    "Para terminar: quantos dias tem o roteiro que estamos montando e para qual cidade ele é?",
]

def conversa_com_contexto(chat, turnos, max_new_tokens=200):
    """Conversa multi-turno: o histórico completo é reenviado ao modelo a cada turno."""
    historico = [{"role": "system", "content": SISTEMA}]
    for texto in turnos:
        historico.append({"role": "user", "content": texto})
        saida = chat(historico, max_new_tokens=max_new_tokens, do_sample=False)
        resposta = saida[0]["generated_text"][-1]["content"]
        historico.append({"role": "assistant", "content": resposta})
        print(f"Usuário: {texto}")
        print(f"Modelo : {resposta}")
        print("-" * 80)
    # Controle: a mesma pergunta final, agora SEM o histórico
    probe = [{"role": "system", "content": SISTEMA},
             {"role": "user", "content": turnos[-1]}]
    resposta = chat(probe, max_new_tokens=120, do_sample=False)[0]["generated_text"][-1]["content"]
    print(f"Controle (pergunta final sem histórico): {resposta}")

In [ ]:
chat_a = pipeline("text-generation", model=MODELO_A, dtype=DTYPE, device=DEVICE)

print(f"===== Modelo A: {MODELO_A} =====\n")
conversa_com_contexto(chat_a, TURNOS)

In [ ]:
del chat_a
liberar_memoria()

chat_b = pipeline("text-generation", model=MODELO_B, dtype=DTYPE, device=DEVICE)

print(f"===== Modelo B: {MODELO_B} =====\n")
conversa_com_contexto(chat_b, TURNOS)

In [ ]:
del chat_b
liberar_memoria()

## **Discussão**

**Manutenção de contexto:** os dois modelos mantiveram o contexto — no 3º turno, ambos responderam corretamente "3 dias, Fortaleza" (o Qwen ainda completou: "capital do estado do Ceará"), informação que só existe no histórico. O controle confirma o papel do contexto: sem o histórico, o Qwen reconhece que não tem como saber e pede mais detalhes, enquanto o TinyLlama **alucina** uma resposta ("o roteiro tem 10 dias e está sendo montado em São Paulo").

**Idioma:** aqui está a maior diferença. O Qwen (multilíngue) conversa em português fluente e obedece ao prompt de sistema; o TinyLlama, treinado quase só em inglês, **ignora a instrução de responder em português** e responde em inglês mesmo com todas as perguntas em português — evidência de que a cobertura multilíngue do pré-treino importa mais do que o porte (1,1B × 1,5B são próximos).

**Qualidade factual:** nenhum dos dois é confiável nos fatos. O Qwen sugere atrações que não existem em Fortaleza ("Praia da Rocha" fica em Portugal; "Boa Viagem" é bairro do Recife), e o TinyLlama recomenda "Cachoeira, a 20 minutos de Fortaleza" (Cachoeira fica na Bahia, a ~1.000 km). Modelos desse porte são fluentes, mas fracos em conhecimento factual local — numa aplicação real de turismo seria necessário ancorar as respostas com RAG (Aula 5).

**Resumo:** o mecanismo de contexto funcionou nos dois; português, aderência ao prompt de sistema e honestidade epistêmica ("não sei") só no Qwen. A escolha de um modelo de chat para aplicações em português não pode olhar só para tamanho e velocidade.

# 2. Task: `Feature-Extration`

Empregue uma task `Feature-Extration` para criar o *embbeding* de duas frases semelhantes em português. Mostre os tokens associados à cada frase e verifique a similaridade dos embbedings das duas frases. Discuta o seu resultado.

<br>

> Opcional: empregue

> ```
> from sklearn.manifold import TSNE
> tsne = TSNE(n_components=2, perplexity=3, random_state=42, n_iter=1000)
> vetores_2d = tsne.fit_transform(todos_os_vetores)
> ```

> para comparar os vetores gerados.

Usamos o [BERTimbau](https://huggingface.co/neuralmind/bert-base-portuguese-cased) (`neuralmind/bert-base-portuguese-cased`), BERT pré-treinado em português pelo grupo NeuralMind.

Desenho do experimento: **duas frases de mesmo sentido escritas com palavras diferentes** (frase 1 e 2) e uma terceira frase de **sentido diferente, mas que compartilha o verbo "subiu"** com a frase 1 — para verificar se o embedding captura semântica ou apenas sobreposição de palavras.

In [ ]:
MODELO_EMB = "neuralmind/bert-base-portuguese-cased"  # BERTimbau

# Modelo pequeno (~110M de parâmetros): CPU é suficiente
extrator = pipeline("feature-extraction", model=MODELO_EMB, device="cpu")

frase_1 = "O gato subiu no telhado da casa."
frase_2 = "O felino escalou o telhado da residência."         # mesmo sentido, palavras diferentes
frase_3 = "A taxa de juros subiu novamente neste trimestre."  # compartilha o verbo "subiu", sentido diferente

# Tokens associados a cada frase (vocabulário WordPiece do BERTimbau)
for f in (frase_1, frase_2, frase_3):
    print(f"{f}\n  tokens: {extrator.tokenizer.tokenize(f)}\n")

In [ ]:
def embedding_frase(frase):
    """Vetores contextuais de todos os tokens -> média (mean pooling) -> um vetor por frase."""
    matriz = np.array(extrator(frase)[0])   # (n_tokens + [CLS] e [SEP], 768)
    return matriz.mean(axis=0)

def similaridade_cosseno(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

matriz_1 = np.array(extrator(frase_1)[0])
print(f"frase_1 vira uma matriz {matriz_1.shape}: um vetor de 768 dims por token (+ [CLS] e [SEP])\n")

emb_1, emb_2, emb_3 = (embedding_frase(f) for f in (frase_1, frase_2, frase_3))

print(f"frase_1 x frase_2 (semelhantes)..............: {similaridade_cosseno(emb_1, emb_2):.4f}")
print(f"frase_1 x frase_3 (só compartilham 'subiu')..: {similaridade_cosseno(emb_1, emb_3):.4f}")
print(f"frase_2 x frase_3 (nada em comum)............: {similaridade_cosseno(emb_2, emb_3):.4f}")

**Opcional (t-SNE).** Para visualizar o espaço de embeddings, adicionamos mais frases em três temas (animais, economia e futebol) e projetamos os vetores de 768 dimensões em 2D com t-SNE, conforme sugerido no enunciado.

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

grupos = {
    "animais":  [frase_1, frase_2,
                 "O cachorro dorme no sofá da sala."],
    "economia": [frase_3,
                 "A inflação acumulada superou a meta do governo.",
                 "O dólar fechou em alta nesta sexta-feira."],
    "futebol":  ["O time marcou dois gols no segundo tempo.",
                 "O goleiro defendeu o pênalti decisivo.",
                 "A torcida lotou o estádio para a final."],
}

frases, rotulos = [], []
for grupo, lista in grupos.items():
    frases.extend(lista)
    rotulos.extend([grupo] * len(lista))

todos_os_vetores = np.vstack([embedding_frase(f) for f in frases])

tsne = TSNE(n_components=2, perplexity=3, random_state=42, max_iter=1000)
vetores_2d = tsne.fit_transform(todos_os_vetores)

CORES  = {"animais": "#2a78d6", "economia": "#eb6834", "futebol": "#1baf7a"}
MARCAS = {"animais": "o", "economia": "s", "futebol": "^"}

fig, ax = plt.subplots(figsize=(8, 5.5))
for grupo in grupos:
    idx = [i for i, r in enumerate(rotulos) if r == grupo]
    ax.scatter(vetores_2d[idx, 0], vetores_2d[idx, 1],
               c=CORES[grupo], marker=MARCAS[grupo], s=90,
               edgecolors="white", linewidths=1.5, label=grupo, zorder=3)
for i in range(len(frases)):
    ax.annotate(str(i + 1), (vetores_2d[i, 0], vetores_2d[i, 1]),
                xytext=(7, 5), textcoords="offset points",
                fontsize=9, color="#52514e")
ax.set_title("t-SNE dos embeddings de frases (BERTimbau, mean pooling)")
ax.set_xlabel("dimensão t-SNE 1")
ax.set_ylabel("dimensão t-SNE 2")
ax.legend(title="tema")
ax.grid(True, color="#e6e4df", linewidth=0.7)
ax.set_axisbelow(True)
for lado in ("top", "right"):
    ax.spines[lado].set_visible(False)
plt.tight_layout()
plt.show()

for i, f in enumerate(frases, 1):
    print(f"{i}. [{rotulos[i-1]}] {f}")

## **Discussão**

**Tokens:** o vocabulário WordPiece do BERTimbau mantém palavras frequentes do português inteiras ("gato", "subiu", "telhado", "casa") e quebra as menos frequentes em subpalavras ("fel"+"##ino", "escal"+"##ou", "tri"+"##mestre"). Cada frase vira uma matriz de vetores contextuais de 768 dimensões — um por token, mais [CLS] e [SEP] — que reduzimos a um único vetor por frase com *mean pooling*.

**Similaridade:** o resultado confirma que o embedding captura semântica, e não apenas sobreposição de palavras:

| Par | O que têm em comum | Cosseno |
|--|--|--|
| frase 1 × frase 2 | mesmo sentido, palavras diferentes | **≈ 0,91** |
| frase 1 × frase 3 | só o verbo "subiu" | ≈ 0,56 |
| frase 2 × frase 3 | nada | ≈ 0,54 |

As duas frases sinônimas ficam muito acima dos demais pares — e o par que compartilha "subiu" fica praticamente no nível do par sem palavra alguma em comum: o modelo distingue "o gato **subiu** no telhado" de "a taxa de juros **subiu**", ou seja, o vetor de cada token é **contextual**, não fixo como no Word2Vec (Aula 2).

**t-SNE:** a projeção 2D agrupa as 9 frases pelos 3 temas (animais, economia, futebol), confirmando visualmente que a vizinhança no espaço de embeddings reflete o assunto. Duas ressalvas: com 9 pontos e `perplexity=3` o t-SNE é só ilustrativo (distâncias globais de t-SNE não são interpretáveis), e cossenos de BERT "cru" tendem a ser altos em geral (anisotropia do espaço) — para similaridade em produção, modelos *sentence-transformers* ajustados para esse fim calibram melhor o espaço.

# 3. Task: `Table-Question-Answer`

Escolha uma tabelas de dados aberta de sua preferência e explore duas ou três questões sobre esses dados empregando um modelo de  `Table-Question-Answer`. Discuta os resultados.

Usamos o [TAPAS](https://huggingface.co/google/tapas-base-finetuned-wtq) (`google/tapas-base-finetuned-wtq`): um BERT estendido com embeddings de linha/coluna que, em vez de gerar texto, **seleciona células da tabela** e sugere um **operador de agregação** (NONE, SUM, AVERAGE, COUNT). Foi ajustado no conjunto WikiTableQuestions (em inglês).

Dados abertos escolhidos: **os 10 estados mais populosos do Brasil no Censo Demográfico 2022 do [IBGE](https://censo2022.ibge.gov.br/)**. Exploramos quatro tipos de pergunta — busca simples, seleção múltipla, agregação — e, ao final, um teste do limite de idioma do modelo (perguntas em português × inglês).

In [ ]:
# Fonte: IBGE — Censo Demográfico 2022 (https://censo2022.ibge.gov.br/)
dados = {
    "Estado": ["São Paulo", "Minas Gerais", "Rio de Janeiro", "Bahia", "Paraná",
               "Rio Grande do Sul", "Pernambuco", "Ceará", "Pará", "Santa Catarina"],
    "Região": ["Sudeste", "Sudeste", "Sudeste", "Nordeste", "Sul",
               "Sul", "Nordeste", "Nordeste", "Norte", "Sul"],
    "População 2022": ["44411238", "20539989", "16055174", "14141626", "11444380",
                       "10882965", "9058931", "8794957", "8116132", "7610361"],
}

# O TAPAS exige que todas as células sejam strings
tabela = pd.DataFrame(dados).astype(str)
tabela

In [ ]:
MODELO_TQA = "google/tapas-base-finetuned-wtq"

tqa = pipeline("table-question-answering", model=MODELO_TQA, device="cpu")

perguntas = [
    "Which state has the largest population?",                          # busca simples
    "Which states belong to the Sudeste region?",                       # seleção múltipla
    "What is the average population of the states in the Sul region?",  # agregação (média)
    "What is the total population of the states in the Nordeste region?",  # agregação (soma)
]

for p in perguntas:
    r = tqa(table=tabela, query=p)
    print(f"Pergunta : {p}")
    print(f"Resposta : {r['answer']}")
    print(f"           agregador={r['aggregator']}, células={r['cells']}\n")

In [ ]:
# O TAPAS devolve as CÉLULAS e o AGREGADOR, mas não executa a conta.
# Para as perguntas de agregação, calculamos o valor a partir das células escolhidas:

r = tqa(table=tabela, query="What is the average population of the states in the Sul region?")
valores = [int(c) for c in r["cells"]]
print(f"Média da população da região Sul..: {sum(valores)/len(valores):,.0f} habitantes "
      f"(agregador sugerido: {r['aggregator']})")

r = tqa(table=tabela, query="What is the total population of the states in the Nordeste region?")
valores = [int(c) for c in r["cells"]]
print(f"Soma da população do Nordeste.....: {sum(valores):,} habitantes "
      f"(agregador sugerido: {r['aggregator']})")

In [ ]:
# O TAPAS foi treinado somente em INGLÊS. A mesma pergunta nas duas línguas:
for p in ["Qual estado tem a menor população?",
          "Which state has the smallest population?"]:
    r = tqa(table=tabela, query=p)
    print(f"{p}\n  -> {r['answer']}\n")

del tqa
liberar_memoria()

## **Discussão**

**O que funcionou:** a busca simples ("largest population" → São Paulo) e a seleção múltipla ("which states belong to the Sudeste region" → São Paulo, Minas Gerais, Rio de Janeiro) vieram exatas. Na média da região Sul, o modelo escolheu exatamente as 3 células corretas (PR, RS e SC) **e** o agregador AVERAGE → ≈ 9.979.235 habitantes.

**Limitações encontradas:**
1. o TAPAS **não executa a aritmética** — ele devolve células + agregador, e a conta fica para o pós-processamento (por isso a célula de cálculo manual);
2. na soma do Nordeste, as células vieram corretas (BA, PE e CE), mas o agregador sugerido foi **COUNT em vez de SUM** — erro típico do TAPAS-base; a soma correta, calculada manualmente, é 31.995.514 habitantes;
3. o teste de idioma é contundente: a mesma pergunta respondida **certo em inglês** ("smallest population" → Santa Catarina, correto dentro da tabela) vem **errada em português** ("menor população" → São Paulo, que é justamente a *maior* — aparente viés pela 1ª linha quando o modelo não entende a pergunta). Coerente com o treino do TAPAS, feito somente em inglês (WikiTableQuestions).

**Conclusão:** para dados abertos brasileiros, o TAPAS funciona bem como *selecionador de células* com perguntas em inglês e tabelas pequenas (o limite de 512 tokens restringe o tamanho da tabela). Em uma aplicação real, as alternativas seriam traduzir as perguntas, usar um modelo multilíngue, ou serializar a tabela no prompt de um LLM instruct (como o Qwen da Task 1).

# 4. Task: uma de sua preferência  

 Implmente uma taks de sua preferência não empregada sem sala de aula. Discuta os resultados.

 Sugestão: **Reconhecimento de Entidades Nomeadas (NER)**

Task escolhida: **Reconhecimento de Entidades Nomeadas** (`token-classification`), não empregada em sala.

A motivação vem do nosso projeto da disciplina (qualidade de laudos médicos): um NER que encontre **pessoas, instituições, datas e locais** é a base de um verificador automático de anonimização de laudos. Comparamos dois modelos abertos sobre os mesmos textos — um trecho no estilo de um laudo/prontuário e uma notícia:

| | Modelo | Base | Treino |
|--|--|--|--|
| 1 | [`pierreguillou/ner-bert-base-cased-pt-lenerbr`](https://huggingface.co/pierreguillou/ner-bert-base-cased-pt-lenerbr) | BERTimbau (PT) | LeNER-Br (textos jurídicos em português) |
| 2 | [`Davlan/bert-base-multilingual-cased-ner-hrl`](https://huggingface.co/Davlan/bert-base-multilingual-cased-ner-hrl) | mBERT (multilíngue) | notícias em 10 línguas |

In [ ]:
texto_laudo = ("A paciente Maria Aparecida dos Santos foi internada no Hospital das Clínicas "
               "de São Paulo no dia 12 de março de 2024, sob responsabilidade do Dr. Carlos "
               "Eduardo Lima, e transferida depois para o Instituto do Coração.")

texto_noticia = ("A Petrobras anunciou em Brasília que o presidente Luiz Inácio Lula da Silva "
                 "visitará a refinaria de Paulínia em dezembro, acompanhado de técnicos do IBGE.")

def mostrar_entidades(ner, texto):
    print(f"TEXTO: {texto}\n")
    for e in ner(texto):
        print(f"  {e['entity_group']:<14} {e['word']!r:<40} score={e['score']:.3f}")
    print()

In [ ]:
ner_pt = pipeline("token-classification",
                  model="pierreguillou/ner-bert-base-cased-pt-lenerbr",
                  aggregation_strategy="simple", device="cpu")

print("===== Modelo 1: BERTimbau + LeNER-Br (português) =====\n")
mostrar_entidades(ner_pt, texto_laudo)
mostrar_entidades(ner_pt, texto_noticia)

In [ ]:
ner_multi = pipeline("token-classification",
                     model="Davlan/bert-base-multilingual-cased-ner-hrl",
                     aggregation_strategy="simple", device="cpu")

print("===== Modelo 2: mBERT multilíngue (10 línguas) =====\n")
mostrar_entidades(ner_multi, texto_laudo)
mostrar_entidades(ner_multi, texto_noticia)

del ner_pt, ner_multi
liberar_memoria()

## **Discussão**

Os dois modelos encontram as **pessoas** com score ≈ 1,0 nos dois textos — mas as diferenças aparecem nos detalhes, e são instrutivas:

- **Datas:** só o modelo em português (LeNER-Br) tem a classe TEMPO e capturou "12 de março de 2024" e "dezembro". O multilíngue não marcou data alguma — para anonimização de laudos (datas de internação!), isso é eliminatório.
- **Instituições:** o modelo PT classificou "Hospital das Clínicas de São Paulo" e "Instituto do Coração" como ORGANIZACAO; o multilíngue marcou ambos como LOC — defensável, porém menos útil para auditoria.
- **Fragmentação:** em compensação, o modelo PT devolveu "Paulínia" quebrada em dois pedaços ("Paul" + "##ínia") — artefato da agregação de subpalavras (`aggregation_strategy="simple"`) sobre uma palavra rara — enquanto o multilíngue devolveu a entidade inteira. Trocar para `aggregation_strategy="first"` ou `"max"` atenua o problema.

**Ligação com o nosso projeto:** o primeiro texto simula o cabeçalho de um laudo. O NER encontrou exatamente os campos sensíveis (paciente, médico, hospital, data) — a base de um verificador automático de anonimização de laudos médicos (ver Task 5). A lição da comparação: **domínio e língua do fine-tuning importam mais do que a arquitetura** — o ideal em produção seria um modelo ajustado em textos clínicos em português, ou o ensemble dos dois.

# 5. Next  

Discuta com o grupo e resuma uma ideia de solução que vocês implementariam com o que aprendeu desses modelos. Procure explorar algo do seu trabalho, ou relacionado ao seu tema de pesquisa. Busque uma implementação o mais original possível.



In [ ]:
# NÃO HÁ NECESSIDADE DE IMPLEMENTAÇÃO AQUI

## **Discussão**

### Ideia: **LaudoCheck** — triagem automática de qualidade e privacidade de laudos médicos

A ideia conecta esta prática ao nosso projeto da disciplina (avaliação da qualidade de laudos médicos com LLMs abertos): um laudo em texto livre entra; sai um painel de qualidade com quatro verificações, cada uma reutilizando uma task exercitada neste notebook.

| Verificação | Task / modelo | Como |
|--|--|--|
| 1. Anonimização (LGPD) | `token-classification` — BERTimbau NER (Task 4) | varre o laudo atrás de PESSOA, ORGANIZACAO, TEMPO e LOCAL remanescentes; qualquer ocorrência em um laudo "anonimizado" gera alerta |
| 2. Completude estrutural | `zero-shot-classification` — mDeBERTa multilíngue (visto em sala) | classifica cada parágrafo em {indicação clínica, técnica, achados, impressão diagnóstica}; seção ausente gera alerta |
| 3. Nexo achados × conclusão | `feature-extraction` — BERTimbau (Task 2) | similaridade de cosseno entre o embedding dos "achados" e o da "impressão"; conclusão semanticamente distante dos achados sugere inconsistência |
| 4. Extração estruturada + auditoria | `text-generation` — Qwen (Task 1) + `table-question-answering` (Task 3) | o modelo instruct extrai campos do laudo (exame, região anatômica, achado principal, CID) para uma tabela; o acervo é então auditado com perguntas em linguagem natural |

**Relevância:** dupla leitura e auditoria de laudos são caras e lentas. Uma triagem automática 100% com modelos abertos — executável em Colab e **sem enviar dados a APIs externas**, essencial com dados de saúde — permite priorizar a revisão humana nos laudos com alerta.

**Avaliação (pesquisa):** aplicar o pipeline ao MIMIC-IV-Note (resumos de alta / laudos radiológicos) e a laudos sintéticos em português, medindo precisão/revocação de cada verificação contra anotação manual, além das métricas semânticas da aula (BERTScore entre achados e conclusão).

**Extensão original (multimodal):** em sala vimos `image-segmentation` sobre imagens DICOM; a evolução natural do LaudoCheck é cruzar o texto com a própria imagem — ex.: o laudo menciona "nódulo no lobo superior direito", mas a segmentação da imagem não encontra estrutura compatível → alerta de divergência laudo × imagem.